# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv


In [2]:
import dask.dataframe as dd

c:\Users\Marsh\miniconda3\envs\marsham_env\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob
file_names=glob(os.path.join(os.getenv('PRICE_DATA'),"**","*.parquet"),recursive=True)

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
dd_prices=dd.read_parquet(file_names)

dd_prices_shift=dd_prices.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(
        close_lag_1=x['Close'].shift(1),
        adj_close_lag_1=x['Adj Close'].shift(1)
        )
)

C:\Users\Marsh\AppData\Local\Temp\ipykernel_27592\4042061177.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_prices_shift=dd_prices.groupby('ticker', group_keys=False).apply(


In [5]:
dd_feat=dd_prices_shift.assign(
    returns=dd_prices_shift['Close']/dd_prices_shift['close_lag_1'] -1,
    hi_lo_range=dd_prices_shift["High"]-dd_prices_shift['Low']
)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
dd_feat.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,close_lag_1,adj_close_lag_1,returns,hi_lo_range
1005,2018-07-03,37.849998,38.240002,37.419998,37.930000,37.930000,507700.0,FWONK.csv,FWONK,2018,NaN,NaN,NaN,0.820004
1006,2018-07-05,37.980000,38.290001,37.430000,37.610001,37.610001,759300.0,FWONK.csv,FWONK,2018,37.930000,37.930000,-0.008437,0.860001
1007,2018-07-06,37.810001,38.680000,37.580002,38.639999,38.639999,822700.0,FWONK.csv,FWONK,2018,37.610001,37.610001,0.027386,1.099998
1008,2018-07-09,38.660000,39.349998,38.470001,38.680000,38.680000,830900.0,FWONK.csv,FWONK,2018,38.639999,38.639999,0.001035,0.879997
1009,2018-07-10,38.599998,39.139999,38.330002,39.080002,39.080002,1423500.0,FWONK.csv,FWONK,2018,38.680000,38.680000,0.010341,0.809998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1547,2019-12-24,52.009998,52.439999,51.810001,52.430000,52.242874,372200.0,ESNT.csv,ESNT,2019,52.070000,51.884159,0.006914,0.629997
1548,2019-12-26,52.549999,52.619999,51.650002,51.810001,51.625088,388200.0,ESNT.csv,ESNT,2019,52.430000,52.242874,-0.011825,0.969997
1549,2019-12-27,51.980000,52.599998,51.750000,52.279999,52.093407,1128400.0,ESNT.csv,ESNT,2019,51.810001,51.625088,0.009072,0.849998
1550,2019-12-30,52.480000,52.480000,51.790001,51.840000,51.654980,335100.0,ESNT.csv,ESNT,2019,52.279999,52.093407,-0.008416,0.689999


c:\Users\Marsh\miniconda3\envs\marsham_env\lib\site-packages\pandas\core\frame.py:717: DeprecationWarning: Passing a BlockManager to DataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
c:\Users\Marsh\miniconda3\envs\marsham_env\lib\site-packages\dask\dataframe\groupby.py:210: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)


In [7]:
dd_feat=dd_feat.groupby('ticker').apply(
    lambda x: x.assign(returns_moving_avg=x['returns'].rolling(10).mean())
)


C:\Users\Marsh\AppData\Local\Temp\ipykernel_27592\4282671690.py:1: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat=dd_feat.groupby('ticker').apply(


In [8]:
dd_feat.compute()

Date       Open       High        Low      Close  Adj Close  \
ticker                                                                          
FWONK  628  2017-01-03  31.590000  32.049999  31.290001  31.500000  31.500000   
       629  2017-01-04  31.680000  31.910000  31.090000  31.790001  31.790001   
       630  2017-01-05  31.900000  32.279999  31.459999  32.119999  32.119999   
       631  2017-01-06  32.330002  32.529999  31.610001  31.650000  31.650000   
       632  2017-01-09  31.150000  31.500000  30.260000  30.660000  30.660000   
...                ...        ...        ...        ...        ...        ...   
ESNT   1170 2018-06-26  36.299999  36.779999  35.869999  36.430000  36.087914   
       1171 2018-06-27  36.419998  36.580002  35.580002  35.820000  35.483639   
       1172 2018-06-28  35.820000  36.320000  35.730000  36.020000  35.681759   
       1173 2018-06-29  36.240002  36.750000  35.779999  35.820000  35.483639   
       1174 2018-07-02  35.619999  36.330002  35.450001  36.320000  35.978947   

               Volume     source ticker  Year  close_lag_1  adj_close_lag_1  \
ticker                                                                        
FWONK  628   261200.0  FWONK.csv  FWONK  2017          NaN              NaN   
       629   122800.0  FWONK.csv  FWONK  2017    31.500000        31.500000   
       630   145400.0  FWONK.csv  FWONK  2017    31.790001        31.790001   
       631   326500.0  FWONK.csv  FWONK  2017    32.119999        32.119999   
       632   260300.0  FWONK.csv  FWONK  2017    31.650000        31.650000   
...               ...        ...    ...   ...          ...              ...   
ESNT   1170  599900.0   ESNT.csv   ESNT  2018    36.330002        35.988853   
       1171  519800.0   ESNT.csv   ESNT  2018    36.430000        36.087914   
       1172  485800.0   ESNT.csv   ESNT  2018    35.820000        35.483639   
       1173  577700.0   ESNT.csv   ESNT  2018    36.020000        35.681759   
       1174  487000.0   ESNT.csv   ESNT  2018    35.820000        35.483639   

              returns  hi_lo_range  returns_moving_avg  
ticker                                                  
FWONK  628        NaN     0.759998                 NaN  
       629   0.009206     0.820000                 NaN  
       630   0.010381     0.820000                 NaN  
       631  -0.014633     0.919998                 NaN  
       632  -0.031280     1.240000                 NaN  
...               ...          ...                 ...  
ESNT   1170  0.002753     0.910000           -0.002014  
       1171 -0.016744     1.000000           -0.002775  
       1172  0.005583     0.590000           -0.002895  
       1173 -0.005552     0.970001           -0.003315  
       1174  0.013959     0.880001           -0.001838  

[326596 rows x 15 columns]

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It was not necessary to convert to a Pandas DataFrame; the calculation could also be done using a Dask DataFrame.
It is recommended to perform such calculations in Dask because it partitions the data and executes computations in parallel. Also, Dask uses lazy evaluation, meaning it does not compute or store results until you explicitly call the .compute() method.
If the dataset is too big to fit in memory, you might need to process it in chunks or keep it as Dask DataFrame to avoid memory errors.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.